# TCN Rainfall Prediction — TensorFlow
## Phase 3 — Temporal Convolutional Network (TCN)
- **Framework:** TensorFlow 2.x (import tensorflow as tf)
- **Input data:** completed_daily_rainfall_cnn_tf.csv (GAN CNN ensemble output)
- **Architecture:** Dilated causal Conv1D blocks with residual connections
- **Task:** Multi-step prediction — forecast next 14 days given past 90 days
- **Metrics:** NRMSE%, NMAE%, R²% (percentage-only outputs)

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)

print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {tf.keras.__version__}")

# ── Constants ──────────────────────────────────────────────────────────────────
STATIONS    = ["nada", "lembing", "reman"]
N_STATIONS  = 3
LOOK_BACK   = 90      # days of history fed to TCN
HORIZON     = 14      # days ahead to predict
BATCH_SIZE  = 64
N_EPOCHS    = 200
PATIENCE    = 30      # early stopping
LR          = 1e-3
SEED        = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

---
## 1. Load Completed Dataset (GAN Output)

In [ ]:
CSV_PATH = "../../data/processed/completed_daily_rainfall_cnn_tf.csv"
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {list(df.columns)}")
print(f"Date range: {df['date'].min().date()}  to  {df['date'].max().date()}")
print(f"NaN total : {df[STATIONS].isna().sum().sum()}")
print()

imp_flags = [s + '_imputed' for s in STATIONS]
for s, f in zip(STATIONS, imp_flags):
    n_imp = df[f].sum() if f in df.columns else 0
    print(f"  {s:<12}: {n_imp:>3} imputed  | mean={df[s].mean():.2f}mm  max={df[s].max():.1f}mm")

df.head()

### 1.1 Time Series Overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 9), sharex=True)
labels = ["Ldg. Nada", "Sg. Lembing", "Ldg. Kuala Reman"]
colors = ["#2196F3", "#4CAF50", "#FF9800"]

for ax, s, label, col in zip(axes, STATIONS, labels, colors):
    f = s + "_imputed"
    obs_mask = df[f] == 0 if f in df.columns else pd.Series([True]*len(df))
    ax.plot(df["date"], df[s], color=col, linewidth=0.5, alpha=0.8, label="Observed")
    if f in df.columns:
        imp_idx = df[f] == 1
        ax.scatter(df.loc[imp_idx, "date"], df.loc[imp_idx, s],
                   color="red", s=6, zorder=5, label="GAN-imputed", alpha=0.6)
    ax.set_ylabel("Rainfall (mm)")
    ax.set_title(label, fontweight="bold")
    ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Date")
plt.suptitle("Completed Daily Rainfall — 3 Stations (2009–2025)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../../figures/tcn/tcn_input_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Preprocessing
### 2.1 Log1p + MinMax Normalisation

In [ ]:
data_raw  = df[STATIONS].values.astype(np.float32)
data_log  = np.log1p(data_raw)

scaler    = MinMaxScaler((0, 1))
data_norm = scaler.fit_transform(data_log)
data_norm = data_norm.astype(np.float32)

print(f"Shape      : {data_norm.shape}")
print(f"Log range  : [{data_log.min():.3f}, {data_log.max():.3f}]")
print(f"Norm range : [{data_norm.min():.3f}, {data_norm.max():.3f}]")

def inv_transform(arr):
    orig = scaler.inverse_transform(arr.reshape(-1, N_STATIONS))
    return np.maximum(np.expm1(orig), 0.0).astype(np.float32)

### 2.2 Sliding Window Sequences

Each sample: past 90 days → next 14 days (all 3 stations simultaneously).

In [ ]:
def make_sequences(data, look_back, horizon):
    X, Y = [], []
    for i in range(len(data) - look_back - horizon + 1):
        X.append(data[i : i + look_back])
        Y.append(data[i + look_back : i + look_back + horizon])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_all, Y_all = make_sequences(data_norm, LOOK_BACK, HORIZON)
print(f"X shape : {X_all.shape}  (samples, look_back, stations)")
print(f"Y shape : {Y_all.shape}  (samples, horizon, stations)")

N = len(X_all)
n_train = int(N * 0.70)
n_val   = int(N * 0.15)
n_test  = N - n_train - n_val

X_tr, Y_tr = X_all[:n_train],          Y_all[:n_train]
X_va, Y_va = X_all[n_train:n_train+n_val], Y_all[n_train:n_train+n_val]
X_te, Y_te = X_all[n_train+n_val:],    Y_all[n_train+n_val:]

print(f"\nTrain  : {len(X_tr):,}  ({100*len(X_tr)/N:.1f}%)")
print(f"Val    : {len(X_va):,}  ({100*len(X_va)/N:.1f}%)")
print(f"Test   : {len(X_te):,}  ({100*len(X_te)/N:.1f}%)")

---
## 3. TCN Architecture

A **Temporal Convolutional Network** uses 1D convolutions with:
- **Causal padding** — only looks at past timesteps (no data leakage)
- **Exponential dilation** — receptive field doubles each layer (1→2→4→8→16→32)
- **Residual connections** — skip connections for gradient flow

**Receptive field** = 2 × (2^n_layers − 1) × kernel_size + 1 per block

In [ ]:
def tcn_block(x, filters, kernel_size, dilation_rate, dropout_rate=0.2):
    residual = x
    # Dilated causal conv (pass 1)
    out = tf.keras.layers.Conv1D(
        filters, kernel_size, padding="causal",
        dilation_rate=dilation_rate, activation="relu",
        kernel_initializer="he_normal"
    )(x)
    out = tf.keras.layers.LayerNormalization()(out)
    out = tf.keras.layers.Dropout(dropout_rate)(out)
    # Dilated causal conv (pass 2)
    out = tf.keras.layers.Conv1D(
        filters, kernel_size, padding="causal",
        dilation_rate=dilation_rate, activation="relu",
        kernel_initializer="he_normal"
    )(out)
    out = tf.keras.layers.LayerNormalization()(out)
    out = tf.keras.layers.Dropout(dropout_rate)(out)
    # Residual projection if channel dims differ
    if residual.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(filters, 1, padding="same")(residual)
    return tf.keras.layers.Add()([residual, out])

def build_tcn(look_back, n_stations, horizon,
              filters=64, kernel_size=3,
              n_blocks=4, dropout=0.2):
    inp = tf.keras.Input((look_back, n_stations), name="rainfall_input")
    x   = inp
    for b in range(n_blocks):
        dilation = 2 ** b
        x = tcn_block(x, filters, kernel_size, dilation, dropout)
    # Global context from last timestep
    x   = tf.keras.layers.Lambda(lambda t: t[:, -1, :])(x)
    x   = tf.keras.layers.Dense(128, activation="relu")(x)
    x   = tf.keras.layers.Dropout(dropout)(x)
    # Output: horizon × stations flattened, then reshaped
    out = tf.keras.layers.Dense(horizon * n_stations)(x)
    out = tf.keras.layers.Reshape((horizon, n_stations))(out)
    return tf.keras.Model(inp, out, name="TCN_Rainfall")

model = build_tcn(LOOK_BACK, N_STATIONS, HORIZON,
                  filters=64, kernel_size=3, n_blocks=4, dropout=0.2)
model.summary()

# Receptive field calculation
rf = 1 + sum(2 * (2**b - 1) * 3 for b in range(4))
print(f"\nReceptive field : ~{rf} days (look-back = {LOOK_BACK} days)")

---
## 4. Training

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(LR, N_EPOCHS * len(X_tr) // BATCH_SIZE, alpha=1e-4)
model.compile(
    optimizer=tf.keras.optimizers.Adam(lr_schedule),
    loss="mse",
    metrics=["mae"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "../../models/tcn_best.keras", monitor="val_loss",
        save_best_only=True, verbose=0
    )
]

history = model.fit(
    X_tr, Y_tr,
    validation_data=(X_va, Y_va),
    epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

stopped_at = len(history.history["loss"])
print(f"\nStopped at epoch : {stopped_at}")
print(f"Best val loss    : {min(history.history['val_loss']):.6f}")

### 4.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history["loss"],     label="Train MSE")
axes[0].plot(history.history["val_loss"], label="Val MSE")
axes[0].set_title("Loss (MSE)", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].plot(history.history["mae"],     label="Train MAE")
axes[1].plot(history.history["val_mae"], label="Val MAE")
axes[1].set_title("MAE (normalised scale)", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE")
axes[1].legend()

plt.suptitle("TCN Training Curves", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../figures/tcn/tcn_tf_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Evaluation — Percentage Metrics

All metrics reported as percentages: NRMSE% = (RMSE / mean_observed) × 100, NMAE% = (MAE / mean_observed) × 100, R²%.

In [ ]:
def pct_metrics(y_true_norm, y_pred_norm, split_label):
    yt = inv_transform(y_true_norm.reshape(-1, N_STATIONS))
    yp = inv_transform(y_pred_norm.reshape(-1, N_STATIONS))
    print(f"\n{'='*65}")
    print(f"  {split_label}")
    print(f"{'='*65}")
    print(f"  {'Station':<14} {'NRMSE%':>8} {'NMAE%':>8} {'R²%':>8}")
    print(f"  {'-'*44}")
    all_t, all_p = [], []
    res = {}
    for j, s in enumerate(STATIONS):
        t = yt[:, j]; p = yp[:, j]
        rmse = np.sqrt(mean_squared_error(t, p))
        mae  = mean_absolute_error(t, p)
        r2   = r2_score(t, p)
        mo   = t.mean()
        res[s] = (rmse, mae, r2, mo)
        print(f"  {s:<14} {rmse/mo*100:>7.1f}% {mae/mo*100:>7.1f}% {r2*100:>7.2f}%")
        all_t.append(t); all_p.append(p)
    at = np.concatenate(all_t); ap = np.concatenate(all_p)
    rmse = np.sqrt(mean_squared_error(at, ap))
    mae  = mean_absolute_error(at, ap)
    r2   = r2_score(at, ap)
    mo   = at.mean()
    print(f"  {'-'*44}")
    print(f"  {'OVERALL':<14} {rmse/mo*100:>7.1f}% {mae/mo*100:>7.1f}% {r2*100:>7.2f}%")
    return rmse, mae, r2, res

Y_va_pred = model.predict(X_va, batch_size=BATCH_SIZE, verbose=0)
Y_te_pred = model.predict(X_te, batch_size=BATCH_SIZE, verbose=0)

va_rmse, va_mae, va_r2, va_res = pct_metrics(Y_va, Y_va_pred, "VALIDATION SET (15%)")
te_rmse, te_mae, te_r2, te_res = pct_metrics(Y_te, Y_te_pred, "TEST SET (15%)")

### 5.1 Accuracy vs Forecast Horizon

Prediction skill typically degrades as the horizon increases — this plot shows how quickly.

In [ ]:
Y_te_orig = inv_transform(Y_te.reshape(-1, N_STATIONS)).reshape(-1, HORIZON, N_STATIONS)
Yp_te_orig = inv_transform(Y_te_pred.reshape(-1, N_STATIONS)).reshape(-1, HORIZON, N_STATIONS)

nrmse_by_day  = []
for h in range(HORIZON):
    row = []
    for j in range(N_STATIONS):
        t = Y_te_orig[:, h, j]; p = Yp_te_orig[:, h, j]
        rmse = np.sqrt(mean_squared_error(t, p)); mo = t.mean()
        row.append(rmse / mo * 100)
    nrmse_by_day.append(np.mean(row))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(1, HORIZON+1), nrmse_by_day, color="#2196F3", alpha=0.8)
ax.set_xlabel("Forecast Day Ahead"); ax.set_ylabel("NRMSE%")
ax.set_title("Forecast Accuracy by Lead Day (Test Set, Avg over Stations)", fontweight="bold")
ax.axhline(nrmse_by_day[0], color="red", linestyle="--", alpha=0.4, label=f"Day-1 = {nrmse_by_day[0]:.1f}%")
ax.legend(); ax.set_xticks(range(1, HORIZON+1))
plt.tight_layout()
plt.savefig("../../figures/tcn/tcn_nrmse_by_day.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Day-1  NRMSE%: {nrmse_by_day[0]:.1f}%")
print(f"Day-7  NRMSE%: {nrmse_by_day[6]:.1f}%")
print(f"Day-14 NRMSE%: {nrmse_by_day[-1]:.1f}%")

### 5.2 Actual vs Predicted (Test Set)

In [ ]:
slabels = ["Ldg. Nada", "Sg. Lembing", "Ldg. Kuala Reman"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, j, s, label in zip(axes, range(N_STATIONS), STATIONS, slabels):
    t  = Y_te_orig[:, :, j].ravel()
    p  = Yp_te_orig[:, :, j].ravel()
    r2 = r2_score(t, p); rmse = np.sqrt(mean_squared_error(t, p)); mo = t.mean()
    ax.scatter(t, p, alpha=0.2, s=5, color="steelblue")
    mv = max(t.max(), p.max()) * 1.1
    ax.plot([0, mv], [0, mv], "r--", lw=1, label="Perfect")
    ax.set_title(f"{label}\nNRMSE={rmse/mo*100:.1f}%   R²={r2*100:.1f}%", fontweight="bold")
    ax.set_xlabel("Actual (mm)"); ax.set_ylabel("Predicted (mm)")
    ax.legend(); ax.set_xlim(0, mv); ax.set_ylim(0, mv)

plt.suptitle("TCN — Actual vs Predicted Rainfall (Test Set)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../../figures/tcn/tcn_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Sample 14-Day Forecast

In [ ]:
n_show = 5
idxs   = np.linspace(0, len(X_te)-1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 1, figsize=(16, n_show*2.5), sharex=False)
for ax, idx in zip(axes, idxs):
    hist  = inv_transform(X_te[idx])[:, 0]          # Nada only for clarity
    true  = Y_te_orig[idx, :, 0]
    pred  = Yp_te_orig[idx, :, 0]
    x_h   = np.arange(-LOOK_BACK, 0)
    x_f   = np.arange(HORIZON)
    ax.plot(x_h, hist[-LOOK_BACK:], color="gray",       lw=0.8, label="History (Nada)")
    ax.bar(x_f, true,   width=0.4, align="edge",   color="#2196F3", alpha=0.7, label="Actual")
    ax.bar(x_f, pred,   width=-0.4, align="edge",  color="#FF5722", alpha=0.7, label="Predicted")
    ax.axvline(0, color="black", lw=0.8, linestyle="--")
    ax.set_ylabel("mm"); ax.legend(loc="upper left", fontsize=7)
    ax.set_title(f"Sample {idx} — 14-day forecast", fontsize=9)
plt.suptitle("TCN Sample Forecasts (Nada Station)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../../figures/tcn/tcn_sample_forecasts.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Save Model & Export

In [ ]:
MODEL_PATH = "../../models/tcn_rainfall_tf.keras"
model.save(MODEL_PATH)

import os
sz = os.path.getsize(MODEL_PATH) / 1024
print(f"Model saved : {MODEL_PATH}  ({sz:.1f} KB)")
print(f"Parameters  : {model.count_params():,}")

# Save test predictions for reporting
test_dates = df["date"].values[n_train + n_val + LOOK_BACK : n_train + n_val + LOOK_BACK + len(Y_te)]
rows = []
for i in range(len(Y_te)):
    for h in range(HORIZON):
        row = {"forecast_date": pd.Timestamp(test_dates[i]) + pd.Timedelta(days=h),
               "horizon_day":   h + 1}
        for j, s in enumerate(STATIONS):
            row[f"{s}_actual"]    = float(Y_te_orig[i, h, j])
            row[f"{s}_predicted"] = float(Yp_te_orig[i, h, j])
        rows.append(row)
df_pred = pd.DataFrame(rows)
df_pred.to_csv("../../predictions/tcn_test_predictions.csv", index=False)
print(f"Predictions : tcn_test_predictions.csv  ({len(df_pred):,} rows)")

---
## Summary

### Architecture

| Component | Detail |
|---|---|
| Framework | TensorFlow 2.21 / Keras 3 |
| Input | 90-day look-back, 3 stations → (batch, 90, 3) |
| TCN blocks | 4 × dilated causal Conv1D (dilations 1, 2, 4, 8) |
| Filters | 64 per block |
| Kernel size | 3 |
| Normalisation | LayerNorm per block |
| Dropout | 0.2 per block + Dense head |
| Output head | Dense(128) → Dense(42) → Reshape(14, 3) |
| Optimiser | Adam + Cosine LR decay (1e-3 → 1e-4) |
| Early stopping | patience=30 on val_loss |

### Input Data
- Source: completed_daily_rainfall_cnn_tf.csv (GAN CNN ensemble output)
- 6,024 rows, zero NaN, 3 stations
- GAN R²=+17.93% — distributional shift <2.2%

### Metric Definitions
- **NRMSE%** = (RMSE ÷ mean_observed) × 100
- **NMAE%**  = (MAE  ÷ mean_observed) × 100
- **R²%**    = coefficient of determination × 100

> Note: NRMSE% for daily rainfall looks large because mean (~8mm) << storm peaks (360mm). This is expected for zero-inflated daily point rainfall.